## TensorFlow 2 and Keras

### Import Keras

In [1]:
import tensorflow.keras as K

### 0. Build a neural network with the Keras library

In [2]:
def build_model(nx, layers, activations, lambtha, keep_prob):
    """
        function that builds a neural network with the Keras library
    """
    model = K.Sequential()

    for i in range(len(layers)):
        model.add(K.layers.Dense(layers[i],
                  activation=activations[i],
                  kernel_regularizer=K.regularizers.L2(lambtha),
                  input_dim=nx))
        # apply dropout except on output layer
        if i != len(layers) - 1 and keep_prob is not None:
            model.add(K.layers.Dropout(1-keep_prob))

    return model


### 1. Build a neural network with the Keras library using functional API

In [3]:
def build_model(nx, layers, activations, lambtha, keep_prob):
    """
        function that builds a neural network with the Keras library

        :param nx: number of input features to the network
        :param layers: list, number nodes in each layer
        :param activations: list, activation functions for each layer
        :param lambtha: L2 regularization parameter
        :param keep_prob: proba node kept for dropout

        :return: keras model
    """
    inputs = K.Input(shape=(nx,))

    x = inputs
    for i in range(len(layers)):
        # add Dense layer
        x = K.layers.Dense(layers[i],
                           activation=activations[i],
                           kernel_regularizer=K.regularizers.L2(lambtha))(x)

        # apply Dropout except last layer
        if i != len(layers) - 1 and keep_prob is not None:
            x = K.layers.Dropout(1 - keep_prob)(x)

    # create model
    model = K.Model(inputs, x)

    return model


### 2. Optimize: Set up Adam optimization for a keras model with categorical crossentropy loss and accuracy metrics

In [4]:
def optimize_model(network, alpha, beta1, beta2):
    """
        function that sets up Adam optimization for a keras model
        with categorical crossentropy loss and accuracy metrics
    """
    Adam_optimizer = K.optimizers.Adam(learning_rate=alpha,
                                       beta_1=beta1,
                                       beta_2=beta2)

    network.compile(optimizer=Adam_optimizer,
                    loss='categorical_crossentropy',
                    metrics=['accuracy'])


### 3. One Hot Encoding. Convert a label vector into a one-hot matrix:

In [5]:
def one_hot(labels, classes=None):
    """
        function that converts a label vector into a one-hot matrix
    """
    return K.utils.to_categorical(labels, classes)

### 4. Train a model using mini-batch gradient descent

In [7]:
def train_model(network, data, labels, batch_size,
                epochs, verbose=True, shuffle=False):
    """
        Function that trains a model using mini-batch gradient descent
    """
    history = network.fit(x=data,
                          y=labels,
                          epochs=epochs,
                          batch_size=batch_size,
                          verbose=verbose,
                          shuffle=shuffle)

    return history


### 5. Train to also analyze validaiton data

In [8]:
def train_model(network, data, labels, batch_size,
                epochs, validation_data=None, verbose=True, shuffle=False):
    """
        Function that trains a model using mini-batch gradient descent
    """
    history = network.fit(x=data,
                          y=labels,
                          epochs=epochs,
                          batch_size=batch_size,
                          validation_data=validation_data,
                          verbose=verbose,
                          shuffle=shuffle)

    return history

### 6. Early stopping.  Train the model using early stopping

### 7.  Learning Rate Decay

Learning rate decay is a **technique used in training neural networks to gradually reduce the learning rate over time**. 

This is often done to fine-tune the training process, allowing the model to converge more effectively as training progresses. 

$$
new\_learning\_rate = \frac{initial\_learning\_rate}{1 + decay\_rate \cdot epochs}
$$

### 8. Save Only the Best

In [9]:
#!/usr/bin/env python3
"""
    Train with Learning Rate Decay
"""

import tensorflow.keras as K


def train_model(network, data, labels, batch_size, epochs, validation_data=None, 
                early_stopping=False, patience=0, learning_rate_decay=False, 
                alpha=0.1, decay_rate=1, 
                save_best=False, filepath=None, verbose=True, shuffle=False):
    """
        Function that trains a model using mini-batch gradient descent
    """
    callback = []
    if early_stopping is True and validation_data is not None:
        early_stop = K.callbacks.EarlyStopping(monitor='val_loss',
                                               patience=patience)

        # add to callback list
        callback.append(early_stop)

    if learning_rate_decay and validation_data:
        # function calculate new learning rate
        def scheduler(epochs):
            lr = alpha / (1 + decay_rate * epochs)
            return lr

        inv_time_decay = K.callbacks.LearningRateScheduler(
            scheduler,
            verbose=1)

        # add to callback list
        callback.append(inv_time_decay)
        
    if save_best:
        save_best_model = K.callbacks.ModelCheckpoint(
            filepath=filepath,
            monitor='val_loss',
            save_best_only=True
        )
        
        callback.append(save_best_model)

    history = network.fit(x=data,
                          y=labels,
                          epochs=epochs,
                          batch_size=batch_size,
                          validation_data=validation_data,
                          callbacks=[callback],
                          verbose=verbose,
                          shuffle=shuffle)

    return history

In [11]:
SEED = 0

import os
os.environ['PYTHONHASHSEED'] = str(SEED)
import random
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
import tensorflow as tf
tf.random.set_seed(SEED)
import tensorflow.keras as K

# Imports


if __name__ == '__main__':
    datasets = np.load('../data/MNIST.npz')
    X_train = datasets['X_train']
    X_train = X_train.reshape(X_train.shape[0], -1)
    Y_train = datasets['Y_train']
    Y_train_oh = one_hot(Y_train)
    X_valid = datasets['X_valid']
    X_valid = X_valid.reshape(X_valid.shape[0], -1)
    Y_valid = datasets['Y_valid']
    Y_valid_oh = one_hot(Y_valid)

    lambtha = 0.0001
    keep_prob = 0.95
    network = build_model(784, [256, 256, 10], ['relu', 'relu', 'softmax'], lambtha, keep_prob)
    alpha = 0.001
    beta1 = 0.9
    beta2 = 0.999
    optimize_model(network, alpha, beta1, beta2)
    batch_size = 64
    epochs = 1000
    train_model(network, X_train, Y_train_oh, batch_size, epochs,
                validation_data=(X_valid, Y_valid_oh), early_stopping=True,
                patience=3, learning_rate_decay=True, alpha=alpha,
                save_best=True, filepath='network1.h5')



Epoch 1: LearningRateScheduler setting learning rate to 0.001.
Epoch 1/1000
782/782 [==============================] - 11s 12ms/step - loss: 0.3284 - accuracy: 0.9206 - val_loss: 0.1848 - val_accuracy: 0.9639 - lr: 0.0010

Epoch 2: LearningRateScheduler setting learning rate to 0.0005.
Epoch 2/1000
  6/782 [..............................] - ETA: 9s - loss: 0.1878 - accuracy: 0.9661 

C:\Users\evisp\anaconda3\Lib\site-packages\keras\src\engine\training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


782/782 [==============================] - 7s 9ms/step - loss: 0.1597 - accuracy: 0.9691 - val_loss: 0.1445 - val_accuracy: 0.9742 - lr: 5.0000e-04

Epoch 3: LearningRateScheduler setting learning rate to 0.0003333333333333333.
Epoch 3/1000
782/782 [==============================] - 9s 11ms/step - loss: 0.1265 - accuracy: 0.9797 - val_loss: 0.1355 - val_accuracy: 0.9768 - lr: 3.3333e-04

Epoch 4: LearningRateScheduler setting learning rate to 0.00025.
Epoch 4/1000
782/782 [==============================] - 8s 10ms/step - loss: 0.1095 - accuracy: 0.9845 - val_loss: 0.1282 - val_accuracy: 0.9788 - lr: 2.5000e-04

Epoch 5: LearningRateScheduler setting learning rate to 0.0002.
Epoch 5/1000
782/782 [==============================] - 9s 11ms/step - loss: 0.0995 - accuracy: 0.9871 - val_loss: 0.1243 - val_accuracy: 0.9797 - lr: 2.0000e-04

Epoch 6: LearningRateScheduler setting learning rate to 0.00016666666666666666.
Epoch 6/1000
384/782 [=============>................] - ETA: 3s - loss: 0.

KeyboardInterrupt: 

### 9. Save and Load Model

### 10. Save and Load Weights

### 11. Save and Load Configuration

### 12. Test

### 13. Predict